# <u>Project 6 - Code Breaking with Statistical Physics

##### __Authors:__ Alex Bell, Lucy Burchell, Hong Chen, Bingxiang Hu, Zoe Morton.

## <u>Introduction

National security, personal privacy and barcodes on the products we buy are some examples of how code breaking is used everyday. In this project, we explore potential methods used to decrypt coded messages with a particular focus on statistical physics. We aim to analyse the effectiveness of these methods by attempting to decrypt twelve encyrpted messages, of which have varying difficulty, thus requiring different methods. 

## <u>Preliminaries

* Just for better formatting:

In [1]:
import nbformat
import requests
import re
from string import ascii_uppercase    #This saves us having to write all the letters out
import math
import random
from collections import Counter
import matplotlib.pyplot as plt

## <u> Section One: Simple Cylclic Shifts

In this section, we will look at a simple cyclic shift of a list of 27 characters. We will continue to use this list throughout this project, and define it using each letter from the alphabet as well as the 'space' character. We will put this in the form of a list in order to perform a smooth cyclic shift for each letter simultaneously.

As an example, we will find the cyclic shift of our first coded message and decode it.

#### Coded Message:

> 1\. GPKZFER JRSER EKWIGIWKWVRZ YZRCWMWCRSEVRYWEWISCRGLIGFJWRGIFYISDD EYRCSEYLSYWRGPKZFEJRVWJ YERGZ CFJFGZPRWDGZSJ QWJRUFVWRIWSVST C KPRN KZR KJREFKSTCWRLJWRFXRJ YE X USEKR EVWEKSK FER KJRCSEYLSYWRUFEJKILUKJRSEVRFTAWUKRFI WEKWVRSGGIFSUZRS DRKFRZWCGRGIFYISDDWIJRNI KWRUCWSIRCFY USCRUFVWRXFIRJDSCCRSEVRCSIYWRJUSCWRGIFAWUKJ

#### Method:

We first started out by identifying the most frequently used character within the coded message. We quickly determinded this to be 'R' which, once compared to the 27 characters available, must be 'space'. From here, we calculated the cyclic shift (from the coded message to our list of characters - so from 'R' to 'space') which was 9.

We then translated this method into python:

In [2]:
#CREATE A PATH TO EASILY REFERENCE CODED MESSAGES
notebook_path = 'CodedMessages.ipynb'
cell_index = 0

try:
    with open(notebook_path, 'r', encoding = 'utf-8') as f:
        nb = nbformat.read(f, as_version = 4)

    cell = nb.cells[cell_index]

    coded_message_01 = cell.source

except FileNotFoundError:
    print('Could not find file.')
except IndexError:
    print('Indexing Error.')

#CHECK THE CORRECT CODED MESSAGE WAS RECEIVED
print(coded_message_01)

GPKZFER JRSER EKWIGIWKWVRZ YZRCWMWCRSEVRYWEWISCRGLIGFJWRGIFYISDD EYRCSEYLSYWRGPKZFEJRVWJ YERGZ CFJFGZPRWDGZSJ QWJRUFVWRIWSVST C KPRN KZR KJREFKSTCWRLJWRFXRJ YE X USEKR EVWEKSK FER KJRCSEYLSYWRUFEJKILUKJRSEVRFTAWUKRFI WEKWVRSGGIFSUZRS DRKFRZWCGRGIFYISDDWIJRNI KWRUCWSIRCFY USCRUFVWRXFIRJDSCCRSEVRCSIYWRJUSCWRGIFAWUKJ


In [3]:
#DEFINE OUR LIST OF ALL 27 CHARACTERS
character_list = ["A" , "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", 
                  "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", " "]

In [4]:
#DEFINE A CYCLIC SHIFT FUNCTION
shift = 9

def cyclic_shift(cipher_letter):

    cipher_index = character_list.index(cipher_letter)
    character_index = (cipher_index + shift) % 27
    
    decipher_letter = character_list[character_index]

    return decipher_letter

#TEST THIS FUNCTION WORKS
cyclic_shift('R')

' '

In [5]:
#Purely for satisfaction, here is a list of all the cyclic shifts:
for letter in character_list:
    print(f'{letter} --> {cyclic_shift(letter)}')

A --> J
B --> K
C --> L
D --> M
E --> N
F --> O
G --> P
H --> Q
I --> R
J --> S
K --> T
L --> U
M --> V
N --> W
O --> X
P --> Y
Q --> Z
R -->  
S --> A
T --> B
U --> C
V --> D
W --> E
X --> F
Y --> G
Z --> H
  --> I


In [6]:
#DEFINE A FUNCTION TO DECODE THE WHOLE MESSAGE
def message_decoder(coded_message):

    deciphered_message = ''

    for i in range(0, len(coded_message),):
        
        current_letter = coded_message[i]

        deciphered_letter = cyclic_shift(current_letter)

        deciphered_message += deciphered_letter

    return deciphered_message
    
#TEST FUNCTION WORKS
message_decoder(coded_message_01)

'PYTHON IS AN INTERPRETED HIGH LEVEL AND GENERAL PURPOSE PROGRAMMING LANGUAGE PYTHONS DESIGN PHILOSOPHY EMPHASIZES CODE READABILITY WITH ITS NOTABLE USE OF SIGNIFICANT INDENTATION ITS LANGUAGE CONSTRUCTS AND OBJECT ORIENTED APPROACH AIM TO HELP PROGRAMMERS WRITE CLEAR LOGICAL CODE FOR SMALL AND LARGE SCALE PROJECTS'

##### The fully decoded message (written with correct punctuation):

>Python is an interpreted, high level, and general purpose programming language. Python's design philosophy EMP hasizes code readability with it's notable use of significant indentation. Its langauge contructs and object-oriented approach aim to help programmers write clear, logical code for small and large-scale projects.

## <u>Section Two: Working with Frequencies

Whilst section one shows that an encypted message built using simple cyclic shifts can be decoded "by hand", other encrytions that use arbitary permutations make the above method impractical. Therefore, we will explore how we could instead use certain statistics of the coded messages. The idea is to permute the characters in our message to a corresponding character in our alphabet and space list based upon the frequencies. We can then track whether the readability of the message increases/improves as we complete these shifts. To do this we need to know two things: the frequencies of the 27 characters generally in the english language and the frequencies of the 27 charcaters in the coded message.

The frequencies of the 27 characters in the english language can be approximated by the frequencies in a large body of english text, for example Moby-dick by Herman Melville. We are only interested in the characters in our list so we must remove all punctuation from the text and capitalise all of the letters. Note, it is also important that the number of spaces does not change during this transformation of the Moby-dick text, as space is one of the 27 characters. 

#### To import the Moby_dick text:

In [7]:
# THIS CELL ONLY NEEDS TO BE RUN ONCE
# Goes to the URL below and writes the contents of the book Moby-dick to a file called moby.txt
# Then saves the text to a list
url = "http://www.gutenberg.org/files/2701/2701-0.txt"
moby = requests.get(url).text

try:                           
    f = open("moby.txt","x")  #Creates the file the first time
except FileExistsError:       #But is skipped if the file already exists
    pass

    
with open("moby.txt","w",errors="ignore") as f:
    f.write(moby) #Write to file
with open("moby.txt","r",errors="ignore") as f:
    moby=f.read() #Read from file

#### Removing unwanted character from the text:

In [8]:
# We want to make all the letters upper case
upper_moby = moby.upper()
# Then we remove anything that isn't an upper case letter, a new line, or a space
moby_new = re.sub(r'[^A-Z\s]', ' ', upper_moby)
# and then remove any additional spaces that weren't in the original text
moby_new = re.sub(r' +', ' ', moby_new)
# finally remove any newlines and replaces them with spaces
moby_new = re.sub(r'\n', ' ', moby_new)

print(moby_new[:1000])

 START OF THE PROJECT GUTENBERG EBOOK      MOBY DICK   OR THE WHALE   BY HERMAN MELVILLE    CONTENTS  ETYMOLOGY   EXTRACTS SUPPLIED BY A SUB SUB LIBRARIAN   CHAPTER LOOMINGS   CHAPTER THE CARPET BAG   CHAPTER THE SPOUTER INN   CHAPTER THE COUNTERPANE   CHAPTER BREAKFAST   CHAPTER THE STREET   CHAPTER THE CHAPEL   CHAPTER THE PULPIT   CHAPTER THE SERMON   CHAPTER A BOSOM FRIEND   CHAPTER NIGHTGOWN   CHAPTER BIOGRAPHICAL   CHAPTER WHEELBARROW   CHAPTER NANTUCKET   CHAPTER CHOWDER   CHAPTER THE SHIP   CHAPTER THE RAMADAN   CHAPTER HIS MARK   CHAPTER THE PROPHET   CHAPTER ALL ASTIR   CHAPTER GOING ABOARD   CHAPTER MERRY CHRISTMAS   CHAPTER THE LEE SHORE   CHAPTER THE ADVOCATE   CHAPTER POSTSCRIPT   CHAPTER KNIGHTS AND SQUIRES   CHAPTER KNIGHTS AND SQUIRES   CHAPTER AHAB   CHAPTER ENTER AHAB TO HIM STUBB   CHAPTER THE PIPE   CHAPTER QUEEN MAB   CHAPTER CETOLOGY   CHAPTER THE SPECKSNYDER   CHAPTER THE CABIN TABLE   CHAPTER THE MAST HEAD   CHAPTER THE QUARTER DECK   CHAPTER SUNSET   CHAPTER D

## Core Question #3

* Question 3 rewording goes here

>3\. (**core**) One simple approach is to use character frequencies. In the sample English text,
count the frequency with which each of the 27 characters appears, and sort them by
frequency. Do the same for the coded message, and try the cipher in which the kth most
common character in the message maps to the kth most common letter in English, for
each k.

#### Coded Message:

>2\. RECFV KUWE VJCRWQFCRCFICYWEZRWKLVJWQF SCRRZ ALVWJLAACFWNATZVW NFWIZRZT FWELYWVCSTWNRWLVTE NBEWZTWHLRWCLRMWS FWJCWHE WUACHWEZJWR WHCVVWT WRCCWTELTWECWHLRWQF S NAYVMWCPKZTCYWTECWJ JCATWTELTWEZVT AWKNGZTTWRWGF LYWGLKUWELYWYZRLQQCLFCYWTEF NBEWTECWY FWJMWK JFLYCWFNRECYWT WTECWTLGVCWVLZYW NTWLVVWTECWRVZQRW SWQLQCFWK ATLZAZABWYLAKZABWJCAWZAWSF ATW SWEZJWLAYWTEFCHWEZJRCVSWZAT WLAWZATFZKLTCWLAYWCVLG FLTCWKLVKNVLTZ AWS FWTH WE NFRWZWHLTKECYWEZJWLRWECWK ICFCYWRECCTWLSTCFWRECCTW SWQLQCFWHZTEWSZBNFCRWLAYWVCTTCFRWR WK JQVCTCVMWLGR FGCYWZAWEZRWTLRUWTELTWECWELYWCIZYCATVMWS FB TTCAWJMWQFCRCAKCWR JCTZJCRWECWHLRWJLUZABWQF BFCRRWLAYWHEZRTVCYWLAYWRLABWLTWEZRWH FUWR JCTZJCRWECWHLRWQNDDVCYWLAYWH NVYWRZTWS FWV ABWRQCVVRWHZTEWLWSNFF HCYWGF HWLAYWLWILKLATWCMCWSZALVVMWECWRQFLABWSF JWEZRWKELZFWHZTEWLWKFMW SWRLTZRSLKTZ AWLAYWHLVUCYWNQWLAYWY HAWTECWF JWFNGGZABWEZRWELAYRWT BCTECFWTECAWECWHF TCWLWV ABWTCVCBFLJWNQ AWLWKLGVCWS FJWZSWJMWLARHCFWT WTEZRWZRWLRWZWE QCWM NWHZVVWELICWLWICFMWQFCTTMWKLRCWT WLYYWT WM NFWK VVCKTZ AWHLTR AWRLZYWECWZWCPQCKTWTELTWHCWRELVVWGCWLGVCWT WB WY HAWT WA FS VUWT J FF HWLAYWT WTLUCW NFWSFZCAYWR JCWICFMWYCSZAZTCWACHRWLRWT WTECWRCKFCTW SWEZRWLAA MLAKCWZWK ASCRRWTELTWZWHLRWSZVVCYWHZTEWKNFZ RZTMWGNTWZWHLRWLHLFCWTELTWE VJCRWVZUCYWT WJLUCWEZRWYZRKV RNFCRWLTWEZRW HAWTZJCWLAYWZAWEZRW HAWHLMWR WZWHLZTCYWNATZVWZTWRE NVYWRNZTWEZJWT WTLUCWJCWZAT WEZRWK ASZYCAKCWGNTWTECFCWHLRWLWYCVLMWZAWTELTWLARHCFZABWTCVCBFLJWLAYWTH WYLMRW SWZJQLTZCAKCWS VV HCYWYNFZABWHEZKEWE VJCRWQFZKUCYWNQWEZRWCLFRWLTWCICFMWFZABW SWTECWGCVVW AWTECWCICAZABW SWTECWRCK AYWTECFCWKLJCWLWVCTTCFWSF JWEZVT AWKNGZTTWLVVWHLRWXNZCTWHZTEWEZJWRLICWTELTWLWV ABWZARKFZQTZ AWELYWLQQCLFCYWTELTWJ FAZABWNQ AWTECWQCYCRTLVW SWTECWRNAYZLVWECWZAKV RCYWLWK QMW SWZTWHEZKEWZRWECFCWFCQF YNKCYWCVRZCWQFCQLFCWT WJCCTWTEMWB YWE VJCRWGCATW ICFWTEZRWBF TCRXNCWSFZCDCWS FWR JCWJZANTCRWLAYWTECAWRNYYCAVMWRQFLABWT WEZRWSCCTWHZTEWLAWCPKVLJLTZ AW SWRNFQFZRCWLAYWYZRJLMWEZRWSLKCWHLRWELBBLFYWHZTEWLAPZCTM

#### Method:

* Maybe briefly explain the ideas you had going into this question
* Basically rewording the question I guess lol

In [9]:
# Count the number of each character that appears in our imported text
characters= ["A" , "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", " "]
for i in range(27):
    frequency= moby_new.count(characters[i])
    print(characters[i], frequency)

A 78224
B 16936
C 22718
D 38311
E 117511
F 20880
G 20875
H 63239
I 65586
J 1088
K 8087
L 42875
M 23325
N 65760
O 69457
P 17443
Q 1570
R 52419
S 64390
T 88391
U 26697
V 8614
W 22264
X 1036
Y 16899
Z 634
  228388


In [10]:
# ALL ENCODED TEXT IS ASSUMED TO BE UPPERCASE

# Defining a new function that takes in a string (could be a large example of English or a specific encoded piece of text)
def get_frequency(text):
    characters = ascii_uppercase + " "
    length = len(text)
    frequency = []

    for char in characters:
        if char == " ":
            frequency.append((char,(text.count(char)+text.count("\n"))/length))    #Calculates the fraction of the text each letter makes up
            
        else:
            frequency.append((char,text.count(char)/length))
            
    # Returns a list containing tuples of each letter and its frequency, sorted by frequency
    return sorted(frequency, key=lambda x: x[1])[::-1]    #Sorts the list by the second entry in each tuple and then reverses it

# Defining a new function that takes in a string to be decoded and a dictionary with the encoded letters as keys and their respective decoded letters as values
def decode(ciphertext,letter_map):
    out = ""
    for char in ciphertext:
        if char != "\n":
            out += letter_map[char]
        else:
            out += letter_map[" "]
    # Returns a decoded string
    return out
    
# Defining a new function that takes in an encoded string and list containing tuples of each letter and its frequency, sorted by frequency (can be got from get_frequency)
def get_letter_map_from_frequency(ciphertext,english_frequencies,verbose = False):
    out = {}
    text_freq = get_frequency(ciphertext)
    print("The frequencies in the text are: \n")
    if verbose:
        for pair in text_freq:
            print(f"{pair[0]}: {pair[1]}")
    for i in range(len(text_freq)):
        encoded_letter = text_freq[i][0]
        english_letter = english_frequencies[i][0]

        out[encoded_letter] = english_letter
    # Returns a dictionary mapping the most common letters in the encoded string with the most common letters in english
    return out

In [11]:
# CREATE A PATH TO EASILY REFERENCE CODED MESSAGES
notebook_path = 'CodedMessages.ipynb'
cell_index = 1

try:
    with open(notebook_path, 'r', encoding = 'utf-8') as f:
        nb = nbformat.read(f, as_version = 4)

    cell = nb.cells[cell_index]

    coded_message_02 = cell.source

except FileNotFoundError:
    print('Could not find file.')
except IndexError:
    print('Indexing Error.')

ciphertext = coded_message_02
import re
moby_clean = moby_new

print(f'The length of our coded message is: {len(coded_message_02)}\n')


# DECODE USING THEIR FREQUENCIES
english_frequencies = get_frequency(moby_new)
letter_map = get_letter_map_from_frequency(ciphertext,english_frequencies)
output= decode(ciphertext,letter_map)
print(output)

print(f'\nThe length of our output message is: {len(output)}')

The length of our coded message is: 1944

The frequencies in the text are: 

ISERLOCV SOLMEI GREIERKED SNI CALM GROWEIINOHAL MAHHER FHTNL OFR KNINTOR SAD LEWT FI ALTSOFPS NT UAI EAIB WOR ME USO VHEU SNM IO UELL TO IEE TSAT SE UAI GROWOFHDLB EQCNTED TSE MOMEHT TSAT SNLTOH CFYNTT I YROAD YACV SAD DNIAGGEARED TSROFPS TSE DOR MB COMRADE RFISED TO TSE TAYLE LAND OFT ALL TSE ILNGI OW GAGER COHTANHNHP DAHCNHP MEH NH WROHT OW SNM AHD TSREU SNMIELW NHTO AH NHTRNCATE AHD ELAYORATE CALCFLATNOH WOR TUO SOFRI N UATCSED SNM AI SE COKERED ISEET AWTER ISEET OW GAGER UNTS WNPFREI AHD LETTERI IO COMGLETELB AYIORYED NH SNI TAIV TSAT SE SAD EKNDEHTLB WORPOTTEH MB GREIEHCE IOMETNMEI SE UAI MAVNHP GROPREII AHD USNITLED AHD IAHP AT SNI UORV IOMETNMEI SE UAI GFJJLED AHD UOFLD INT WOR LOHP IGELLI UNTS A WFRROUED YROU AHD A KACAHT EBE WNHALLB SE IGRAHP WROM SNI CSANR UNTS A CRB OW IATNIWACTNOH AHD UALVED FG AHD DOUH TSE ROM RFYYNHP SNI SAHDI TOPETSER TSEH SE UROTE A LOHP TELEPRAM FGOH A CAYLE WORM NW MB AHIUER 

## Core Question #4

* Question rewording once again por favor
* Also maybe explain somewhere why we chose to swap specific letters, not all of them obviously but some key ones to start us off and the last swap, just the thought process really.

>4\. (**core**) The above may not be sufficient to get the full correct answer, but for the longer
messages it should be close. Write a program to make it easy for you to try swapping the
roles of any two characters (of your choice) and immediately see the effect of the decoded
message. Then, starting from the result above, adjust the cipher by hand until is readable.

In [12]:
# We want a function that swaps all of one character with another, both are specified as the variables of the function
def swap_of_char(string, char_1, char_2):
    # If both letters are the same, the string will be as it originally was
    if char_1 == char_2: 
        return string
    else:
        # Using a temporary character $ to replce all of char_1
        string= string.replace(char_1, "$")
        # Replacing all of char_2 with char_1
        string= string.replace(char_2,char_1)
        # Finally replacing all of temporary character $ with char_2
        string= string.replace("$", char_2)
    return string
# After seeing how this swap has effected the code, you can adjust the cipher by hand:
output= decode(ciphertext,letter_map)
output= swap_of_char(output, "P", "G")
output= swap_of_char(output, "W", "F")
output= swap_of_char(output, "I", "H")
output= swap_of_char(output, "S", "H")
output= swap_of_char(output, "I", "N")
output= swap_of_char(output, "V", "K")
output= swap_of_char(output, "W", "U")
output= swap_of_char(output, "B", "Y")
output= swap_of_char(output, "Q", "X")
swap_of_char(output, "J", "Z")

'SHERLOCK HOLMES PRESERVED HIS CALM PROFESSIONAL MANNER UNTIL OUR VISITOR HAD LEFT US ALTHOUGH IT WAS EASY FOR ME WHO KNEW HIM SO WELL TO SEE THAT HE WAS PROFOUNDLY EXCITED THE MOMENT THAT HILTON CUBITT S BROAD BACK HAD DISAPPEARED THROUGH THE DOR MY COMRADE RUSHED TO THE TABLE LAID OUT ALL THE SLIPS OF PAPER CONTAINING DANCING MEN IN FRONT OF HIM AND THREW HIMSELF INTO AN INTRICATE AND ELABORATE CALCULATION FOR TWO HOURS I WATCHED HIM AS HE COVERED SHEET AFTER SHEET OF PAPER WITH FIGURES AND LETTERS SO COMPLETELY ABSORBED IN HIS TASK THAT HE HAD EVIDENTLY FORGOTTEN MY PRESENCE SOMETIMES HE WAS MAKING PROGRESS AND WHISTLED AND SANG AT HIS WORK SOMETIMES HE WAS PUZZLED AND WOULD SIT FOR LONG SPELLS WITH A FURROWED BROW AND A VACANT EYE FINALLY HE SPRANG FROM HIS CHAIR WITH A CRY OF SATISFACTION AND WALKED UP AND DOWN THE ROM RUBBING HIS HANDS TOGETHER THEN HE WROTE A LONG TELEGRAM UPON A CABLE FORM IF MY ANSWER TO THIS IS AS I HOPE YOU WILL HAVE A VERY PRETTY CASE TO ADD TO YOUR COLLECT

The fully decoded message (with correct punctuation):
>Sherlock Holmes preserved his calm professional manner until our visitor had left us, although it was easy for me, who knew him so well, to see that he was profoundly excited. The moment that Hilton Cubitt’s broad back had disappeared through the door my comrade rushed to the table, laid out all the slips of paper containing dancing men in front of him, and threw himself into an intricate and elaborate calculation. For two hours I watched him as he covered sheet after sheet of paper with figures and letters, so completely absorbed in his task that he had evidently forgotten my presence. Sometimes he was making progress and whistled and sang at his work. Sometimes he was puzzled, and would sit for long spells with a furrowed brow and a vacant eye. Finally he sprang from his chair with a cry of satisfaction, and walked up and down the room rubbing his hands together Then he wrote a long telegram upon a cable form. “If my answer to this is as I hope, you will have a very pretty case to add to your collection, Watson,” said he. “I expect that we shall be able to go down to Norfolk to-morrow, and to take our friend some very definite news as to the secret of his annoyance. I confess that I was filled with curiosity, but I was aware that Holmes liked to make his disclosures at his own time and in his own way, so I waited until it should suit him to take me into his confidence. But there was a delay in that answering telegram, and two days of impatience followed, during which Holmes pricked up his ears at every ring of the bell. On the evening of the second there came a letter from Hilton Cubitt. All was quiet with him, save that a long inscription had appeared that morning upon the pedestal of the sun-dial. He inclosed a copy of it, which is here reproduced, Holmes bent over this grotesque frieze for some minutes, and then suddenly sprang to his feet with an exclamation of surprise and dismay. His face was haggard with anxiety.

## <u>Section Three: Applying Statistical Physics

* Brief run down on what's going on in this section,
* How it relates to the last section for better flow,
* And general ideas going into this section too

## Core Question #5

* Reword the words pls

Since the previous method is less likely to work for later messages (because they are not long enough to get accurate frequency counts), we will use a "Markov model" approach based on statistical physics.

**First** we must record the frequency of:
* Each of the 27 characters.
* Each of the $27^2$ bigrams (pairs of consecutive characters).

**Next**, for each pair of characters $i, j$, we compute the probability that $i$ is followed by $j$ using the formula:

$$p(i, j) := \frac{\text{frequency}(ij) + 1}{\text{frequency}(i)}$$

*(Note: We include a $+1$ 'fudge factor' to avoid zeros).*

**Finally**, for any potential decoded message $m = i_1i_2\dots i_n$, we assign a score $S(m)$ which measures its plausibility as English text:

$$S(m) = \sum_{j=1}^{n-1} \log p(i_j, i_{j+1})$$

**Our Task:**
Write a function that computes $S(m)$ for any message $m$.

**We need to switch and decode the functions used previously, so to streamline, we have just copied all the functions we have defined thus far.**

In [13]:
## Logic and knowledge of the english language tells us certian pairs of letters, for example: 'wj', have zero frequency whereas others such as 'ee' are more frequent.

# Let's recall and define our alphabet 
character_list = ["A" , "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", " "]

def all_bigram_freqs(string,character_list):
    # Defining the dictionary of frequencies
    single_frequencies= {}
    bi_frequencies= {}

    #sets the values for each letter and bigram as 1    
    for char_1 in character_list:
        single_frequencies[char_1] = 0
        
        for char_2 in character_list:
            pair= char_1 + char_2
            bi_frequencies[pair]= 0

    #iterates through the text, updating the single and bigram dictionaries for the current letter and bigram
    for i in range(len(string)-1):
        single_frequencies[string[i]] += 1
        bi_frequencies[string[i:i+2]] += 1
        
    return single_frequencies , bi_frequencies
    
# Note index [0] will give the single frequencies dictionary and index [1] will give the bigram frequencies

# Now we want to define a function that, for a pair of charcters, finds the probability that the first letter is followed by the second letter in the piece of text

def proportion(char_1, char_2, all_bigram_freqs_dicts):
    x= all_bigram_freqs_dicts
    pair= char_1 + char_2
    p= (x[1][pair] + 1 )/(x[0][char_1])
    return p


# The score function S(m) can be defined as
def score(string,all_bigram_freq_dicts):
    
    n = len(string)    # Where n is the total number of characters in the string
    total_log_ps = 0
    
    for j in range(1, n-1):
        
        # Index the string of text to find the jth and j+1th letter
        p_j= proportion(string[j], string[j+1],all_bigram_freq_dicts)
        total_log_ps += math.log(p_j)

    return total_log_ps


## Core Question #6

* Reword again, merci beaucoup

Implement the **Metropolis algorithm**. Start with any guess $m$ at the decrypted message. Repeatedly choose two characters at random and swap their roles to get a new candidate message $m'$.

If $S(m') > S(m)$ then replace $m$ with $m'$. If $S(m') \leq S(m)$ then replace $m$ with $m'$ with probability:

$$\exp \frac{S(m') - S(m)}{T}$$

otherwise just keep $m$. Repeat this for many steps (e.g. 100000 or 1000000). Start by trying $T = 1$.

#### Coded Message:

> 3\. TULOMREMAEKBLRSWCTMWBIHB BTN KKUT NBTULOMRB DWBKSYMBYSWMRDBTULOMRKBJUNNBRMQM NBKE EUKEUT NBUDPSRY EUSDB ISCEBEOMBLN UDEMAEB DWBEO EBUDPSRY EUSDBT DBSPEMDBIMBCKMWBESBIRM XBEOMBTULOMRB PEMRBEOMBWUKTSQMRHBSPBPRMVCMDTHB D NHKUKBIHBEOMB R IBY EOMY EUTU DB DWBLSNHY EOB NBXUDWUB NKSBXDSJDB KB NXUDWCKBUDBEOMBDUDEOBTMDECRHBDM RNHB NNBKCTOBTULOMRKBTSCNWBIMBIRSXMDBIHB DBUDPSRYMWB EE TXMRBKCTOBTN KKUT NBTULOMRKBKEUNNBMDGSHBLSLCN RUEHBESW HBEOSCFOBYSKENHB KBLCZZNMKB NBXUDWUBJRSEMB BISSXBSDBTRHLESFR LOHBMDEUENMWBRUK N OBPUBUKEUXOR GB NBYC YY BY DCKTRULEBPSRBEOMBWMTULOMRUDFBTRHLESFR LOUTBYMKK FMKBJOUTOBWMKTRUIMWBEOMBPURKEBXDSJDBCKMBSPBPRMVCMDTHB D NHKUKB DWBTRHLE D NHKUKBEMTODUVCMKB DBUYLSRE DEBTSDERUICEUSDBSPBUIDB WN DBJ KBSDBK YLNMBKUZMBPSRBCKMBSPBPRMVCMDTHB D NHKUK

In [14]:
#IMPORTING THIS SAME CODED MESSAGE TO REFERENCE SMOOTHLY
notebook_path = 'CodedMessages.ipynb'
cell_index = 2

try:
    with open(notebook_path, 'r', encoding = 'utf-8') as f:
        nb = nbformat.read(f, as_version = 4)

    cell = nb.cells[cell_index]

    message = cell.source

except FileNotFoundError:
    print('Could not find file.')
except IndexError:
    print('Indexing Error.')

#CHECK THE CORRECT CODED MESSAGE WAS RECEIVED
print(f'Our third coded message:\n\n{message}')

Our third coded message:

TULOMREMAEKBLRSWCTMWBIHB BTN KKUT NBTULOMRB DWBKSYMBYSWMRDBTULOMRKBJUNNBRMQM NBKE EUKEUT NBUDPSRY EUSDB ISCEBEOMBLN UDEMAEB DWBEO EBUDPSRY EUSDBT DBSPEMDBIMBCKMWBESBIRM XBEOMBTULOMRB PEMRBEOMBWUKTSQMRHBSPBPRMVCMDTHB D NHKUKBIHBEOMB R IBY EOMY EUTU DB DWBLSNHY EOB NBXUDWUB NKSBXDSJDB KB NXUDWCKBUDBEOMBDUDEOBTMDECRHBDM RNHB NNBKCTOBTULOMRKBTSCNWBIMBIRSXMDBIHB DBUDPSRYMWB EE TXMRBKCTOBTN KKUT NBTULOMRKBKEUNNBMDGSHBLSLCN RUEHBESW HBEOSCFOBYSKENHB KBLCZZNMKB NBXUDWUBJRSEMB BISSXBSDBTRHLESFR LOHBMDEUENMWBRUK N OBPUBUKEUXOR GB NBYC YY BY DCKTRULEBPSRBEOMBWMTULOMRUDFBTRHLESFR LOUTBYMKK FMKBJOUTOBWMKTRUIMWBEOMBPURKEBXDSJDBCKMBSPBPRMVCMDTHB D NHKUKB DWBTRHLE D NHKUKBEMTODUVCMKB DBUYLSRE DEBTSDERUICEUSDBSPBUIDB WN DBJ KBSDBK YLNMBKUZMBPSRBCKMBSPBPRMVCMDTHB D NHKUK


In [15]:
#part 6
#Construct a dictionary of the frequency of every letter and every possible bigram in a sample text
all_bigram_freqs_dicts = all_bigram_freqs(moby_new,character_list)

#Form an algorithim to decode a piece of encoded English text:
    ##Take in a ciphertext, a list of all characters present (in caps), a sample frequency dictionary of bigrams and letters, the number of iterations
    ##to run for (let's call this N), the temperature used in the algorithim (T), a verbose keyword argument which prints the current parrtially decoded 
    ##text in at certain steps and a simulated annealing keyword argument which, when True, allows metropolis to change the T throughout
    ##- then return the decoded text
def metropolis(ciphertext, character_list, all_bigram_freqs_dicts, N, T, verbose = False, simulated_annealing = False):
    n = len(character_list)
    current_string = ciphertext
    T_0 = T

    #For Question 9, we need to record the history of the score after each step:
    score_history = []
    
    #We now need to iterate N times, switching two random letters to create a candidate message, and then take the candidate message according to
    #probabilities defined by the Metropolis Algorothim (based on the Boltzmann Distribution)
    for i in range(N):
        letter_1 = character_list[random.randrange(n)]
        letter_2 = character_list[random.randrange(n)]
        potential_string = swap_of_char(current_string,letter_1,letter_2)
        
        s_current = score(current_string,all_bigram_freqs_dicts)
        s_potential = score(potential_string,all_bigram_freqs_dicts)

        if s_potential > s_current:
            current_string = potential_string
        else:
            r = random.random()
            val = math.exp((s_potential - s_current)/T) #Here T is prop. to the probability - we take a backwards step in the hopes it later improves the text
            if r <= val:
                current_string = potential_string

        #Updating our history record
        score_history.append(s_current)

        #If verbose, we print the current string every 10% through N we get
        if verbose:
            if i%(N/10) == 0:
                print(current_string)
                print(T)

        #if simulated annealing, linearly decrease the T each step (with a slight fudge factor to avoid 0)
        if simulated_annealing:
            T = (N-i+1)*T_0/N

    return current_string, score_history


final_text, score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 30000, 10, verbose = True,simulated_annealing = True)
print(final_text)

TULOMREMAEKBLRSWCTMWBIHB BTN KKUT NBTULOMRB DWBKSYMBYSWMRDBTULOMRKBJUNNBRMQM NBKE EUKEUT NBUDPSRY EUSDB ISCEBEOMBLN UDEMAEB DWBEO EBUDPSRY EUSDBT DBSPEMDBIMBCKMWBESBIRM XBEOMBTULOMRB PEMRBEOMBWUKTSQMRHBSPBPRMVCMDTHB D NHKUKBIHBEOMB R IBY EOMY EUTU DB DWBLSNHY EOB NBXUDWUB NKSBXDSJDB KB NXUDWCKBUDBEOMBDUDEOBTMDECRHBDM RNHB NNBKCTOBTULOMRKBTSCNWBIMBIRSXMDBIHB DBUDPSRYMWB EE TXMRBKCTOBTN KKUT NBTULOMRKBKEUNNBMDGSHBLSLCN RUEHBESW HBEOSCFOBYSKENHB KBLCZZNMKB NBXUDWUBJRSEMB BISSXBSDBTRHLESFR LOHBMDEUENMWBRUK N OBPUBUKEUXOR GB NBYC YY BY DCKTRULEBPSRBEOMBWMTULOMRUDFBTRHLESFR LOUTBYMKK FMKBJOUTOBWMKTRUIMWBEOMBPURKEBXDSJDBCKMBSPBPRMVCMDTHB D NHKUKB DWBTRHLE D NHKUKBEMTODUVCMKB DBUYLSRE DEBTSDERUICEUSDBSPBUIDB WN DBJ KBSDBK YLNMBKUZMBPSRBCKMBSPBPRMVCMDTHB D NHKUK
10
CIPHERTEVTS PRODUCED BY A CLASSICAL CIPHER AND SOME MODERN CIPHERS WILL REXEAL STATISTICAL INFORMATION ABOUT THE PLAINTEVT AND THAT INFORMATION CAN OFTEN BE USED TO BREAK THE CIPHER AFTER THE DISCOXERY OF FREQUENCY ANALYSIS BY THE AR

In [16]:
import nbformat


#finds the specified coded message and returns it if exists
def get_coded_message(num):
    notebook_path = 'CodedMessages.ipynb'
    cell_index = num

    try:
        with open(notebook_path, 'r', encoding = 'utf-8') as f:
            nb = nbformat.read(f, as_version = 4)
    
        cell = nb.cells[cell_index]
    
        coded_message = cell.source
    
    except FileNotFoundError:
        print('Could not find file.')
    except IndexError:
        print('Indexing Error.')

    return coded_message

In [48]:
notebook_path = 'CodedMessages.ipynb'
cell_index = 11

try:
    with open(notebook_path, 'r', encoding = 'utf-8') as f:
        nb = nbformat.read(f, as_version = 4)

    cell = nb.cells[cell_index]

    coded_message_12 = cell.source

except FileNotFoundError:
    print('Could not find file.')
except IndexError:
    print('Indexing Error.')

print(coded_message_12)

RIZOGS BXTPK YWH JQNE LUC MADFV


* Below this is me trying the method of question 6 onto the 12th coded message we are given, which we were thinking could be a different language entirely because these lecturers hate us. I mean because it's more fun that way. But we can also move this to the 'Results' section at the very end

In [49]:
coded_message_twelve, twelfth_score_history = metropolis(coded_message_12, character_list, all_bigram_freqs_dicts, 20000, 1, verbose = True)
print(coded_message_twelve)

RIZOGS BXTPK YWH JQNE LUC MADFV
1
XJOVAG SQUBY MPR WNDF HIZ TCKEL
1
JWNVEG BQUZY TLD AXPR HOF ICKSM
1
KVEZOF QUMPY JXT BRLD ICH WSANG
1
VXCKIG QUDST ZOF WHEM PLY RBJAN
1
ZJVING FEQUS RCK THAM WOD BLYXP
1
GXCKLD SPUTH BZY WOFR MAN JEIVQ
1
ZXCKIN QUPLD GHY JOFR VAS TWEMB
1
QXCKWH VABUN STY RELF OMP JZIDG
1
QXCKWH VEFZY AND JULO BST GRIMP
1
VXCKWF ERSTH AND JUGL PBY QZIMO


## <u>Section Four: Improvements and Analysis

* Insert another general overview of this section here

## Extension Question #7

* Question rewording

> 7\. (**extension**) How can the method be improved? The final few messages will require thinking outside
>the box. Does it help to consider trigrams, or just common ones, or whole words? What
>if the message is in a unknown language? Can the language be detected automatically? Can the method cope with small errors (typos) in the message, or even correct them?

In [27]:
for i in range(3,12):
    final_text,_ = metropolis(get_coded_message(i), character_list, all_bigram_freqs_dicts, 30000, 10 ,simulated_annealing = True)
    print(i+1,":",final_text)

4 : WORKING WITH THEIR EGYPTIAN COLLEAGUES THE DUTCH CONSULTANTS WERE RUNNING NUMBERS BOSKALISS ROLE WAS BASICALLY TO DO THE CALCULATIONS THE SUMS BERDOWSKI SAID SO WHEN THE STERN CAME FREE ON MONDAY MORNING WE CALCULATED THAT WE SHOULD LET TONNES OF WATER BALLAST IN AT THE REAR OF THE VESSEL TO PUSH THE STERN DOWN AND LIFT THE BOW
5 : A MERY CHOICE FONVEY WAS TAVEN ILL AND REQUSED TO EAT IT WAS THOUGHT HOWEMER THAT ITS APPETITE FIGHT BE STIFULATED BY A PINEAPPLE
6 : TATTERDEMALION AND A BUNKETER THERES A THIEF AND A DRAGONFLY TRUMPETER HES MY HERO FAIRY DANDY TICKLING THE FANCY OF HIS LADY FRIEND THE NYMPH IN YELLOW CAN WE SEE THE MASTER STROKE WHAT A QUAERE FELLOW
7 : JINCALY AD HESSESHALYESTARYOLACOWAD ALY A SU
8 : AMANG THE TREES WHERE HUMMING BEES AT BUDS AND FLOWERS WERE HINGING O AULD CALEDON DREW OUT HE DRONE AND TO HER PIPE WAS SINGING O TWAS PIBROCH SANG STRATHSPEYS AND REELS SHE DIRLD THEM AFF FU CLEARLY O WHEN THERE CAM A YELL O FOREIGN SQUEELS THAT DANG HER TAPSALTEERIE O


message 4: Working with their Egyptian colleagues, the Dutch consultants were running numbers. “Boskalis’s role was basically to do the calculations, the sums,” Berdowski said. “So when the stern came free [on Monday] morning, we calculated that we should let 2,000 tonnes of water ballast in at the rear of the vessel, to push the stern down and lift the bow.- from a guardian article on freeing the Ever Given from Suez canal

message 5: A very chpice monkey was taken ill and refused to eat. It was thought however that its appetite might be stimulated by a pineapple- unsure of origin

message 6: Tatterdemalion and the junketer 
           There's a thief and a dragonfly trumpeter, he's my hero
           Fairy dandy tickling the fancy of his lady friend
           The nymph in yellow (can we see the master stroke)
           What a quaere fellow - 
from The Fairy Feller's Master-Stroke by Queen

message 8: Amang the trees, where humming bees, 
At buds and flowers were hinging, O, 
Auld Caledon drew out her drone, 
And to her pipe was singing, O: 
'Twas Pibroch, Sang, Strathspeys, and Reels, 
She dirl'd them aff fu' clearly, O: 
When there cam' a yell o' foreign squeels, 
That dang her tapsalteerie, O. - A Fiddler In The North, robbie burns

In [30]:
# Word analysis is based on the observation that in English text, certain words appear with much higher frequency than others. 
# For example: "THE", "AND", "OF", "TO", "IN", etc.
character_list = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", 
                  "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", " "]

def build_word_frequency_dict(text, min_word_length=2):
    words = text.split()
    word_freq = {}
    for word in words:
        if len(word) >= min_word_length: # Only consider words length greater than 2
            word_freq[word] = word_freq.get(word, 0) + 1
    total_words = sum(word_freq.values())
    
    # Display statistics
    print(f"Built dictionary with {len(word_freq)} unique words")
    print("Top 10 most frequent words:")
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]
    for word, freq in sorted_words:
        probability = freq / total_words
        print(f"  '{word}': {freq} occurrences ({probability:.4f})")
    
    return word_freq, total_words

In [46]:
#creates database of sample texts in several languages
import requests
import re

#gets the text from a link and leaves only the characters we want
def get_text(url,file_name):
    file = requests.get(url).text

    try:                           
        f = open(file_name,"x")  #Creates the file the first time
    except FileExistsError:       #But is skipped if the file already exists
        pass
    
        
    with open(file_name,"w",errors="ignore") as f:
        f.write(file) #Write to file
    with open(file_name,"r",errors="ignore") as f:
        file=f.read() #Read from file
    
    # We want to make all the letters upper case
    file = file.upper()
    # Then we remove anything that isn't an upper case letter, a new line, or a space
    file = re.sub(r'[^A-Z\s]', ' ', file)
    # and then remove any additional spaces that weren't in the original text
    file = re.sub(r' +', ' ', file)
    # finally remove any newlines and replaces them with spaces
    file = re.sub(r'\n', ' ', file)
    file = re.sub(r'\xa0', ' ', file)

    return file

#sets up the different languages and links to texts in those languages, then populates sample_texts with the processed texts
languages = ["english","french","spanish","german","dutch","portuguese"]
country_codes = {"english":"en","french":"fr","spanish":"es","german":"de","dutch":"nl","portuguese":"pt"}
urls = ["http://www.gutenberg.org/files/2701/2701-0.txt",
        "https://www.gutenberg.org/cache/epub/2650/pg2650.txt",
        "https://www.gutenberg.org/cache/epub/2000/pg2000.txt",
        "https://www.gutenberg.org/cache/epub/22367/pg22367.txt",
        "https://www.gutenberg.org/cache/epub/22722/pg22722.txt",
        "https://www.gutenberg.org/cache/epub/3333/pg3333.txt"]
file_names = [x+".txt" for x in languages]
sample_texts = {}

for i,lang in enumerate(languages):
    sample_texts[lang] = get_text(urls[i],file_names[i])


In [47]:
#runs through the chosen languages 
def metropolis_many_languages(text,sample_texts,languages,verbose = False):
    out = (0,-1e10)
    for lang in languages:
        all_bigram_freqs_dicts = all_bigram_freqs(sample_texts[lang],character_list)
        final_text, score_history = metropolis(text, character_list, all_bigram_freqs_dicts, 30000, 10, verbose = False,simulated_annealing = True)
        score = score_history[-1]
        if score >= out[1]:
            out = (final_text,score,lang)
        if verbose:
            print(final_text,score,lang)
    return out

text,_,lang = metropolis_many_languages(get_coded_message(8),sample_texts,languages)

In [56]:
!pip install translate

9197.53s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [55]:
#translates decoded work
import translate

print(lang,":",text)

translator = Translator(to_lang="en",from_lang=country_codes[lang])
translation = translator.translate(text)

print("")
print(translation)

french : JE ME RENDORMAIS ET PARFOIS JE NAVAIS PLUS QUE DE COURTS REVEILS DUN INSTANT LE TEMPS DENTENDRE LES CRAQUEMENTS ORGANIQUES DES BOISERIES DOUVRIR LES YEUX POUR FIXER LE WALEIDOSCOPE DE LOBSCURITE DE GO TER GR CE UNE LUEUR MOMENTAN E DE CONSCIENCE LE SOMMEIL OU ETAIENT PLONG S LES MEUBLES LA CHAMBRE LE TOUT DONT JE NETAIS QUUNE PETITE PARTIE ET A LINSENSIBILITE DUQUEL JE RETOURNAIS VITE MUNIR

I GET TIRED AND SOMETIMES I WEAR MORE THAN SHORT COATINGS FOR A MOMENT THE TIME TO RELAX THE ORGANIC CRACKS OF THE WOODWORK TO OPEN THE EYES TO FIX THE WALEIDOSCOPE OF GO TER 'S LOBSCURITE THIS A MOVING GLOW OF CONSCIOUSNESS THE SLEEP WHERE WERE THE FURNITURE THE ROOM ALL OF WHICH I HAD ONLY A SMALL PART AND TO THE INSENSITIVITY OF WHICH I QUICKLY RETURNED


this is Du Côté de Chez Swann by Marcel Proust

In [ ]:
# Calculate text score based on word frequencies
def word_based_score(text, word_freq_dict, total_words, penalty_weight=2.0):
    score = 0
    words = text.split()
    
    if len(words) == 0:
        return float('-inf')  # Empty text gets worst score
    
    known_words = 0
    unknown_words = 0
    
    for word in words:
        if word in word_freq_dict:
            # Known word: positive contribution based on frequency
            probability = word_freq_dict[word] / total_words
            score += math.log(probability + 1e-10)  # Smoothing to avoid log(0)
            known_words += 1
        else:
            # Unknown word: apply penalty
            score += math.log(1e-10) * penalty_weight
            unknown_words += 1
    
    # Additional reward for high percentage of known words
    if len(words) > 0:
        known_ratio = known_words / len(words)
        score += math.log(known_ratio + 1e-10) * 3
    
    return score

In [ ]:
# Combined scoring using both bigram and word frequencies
def combined_score(text, bigram_freq_dicts, word_freq_dict, total_words, 
                  alpha=0.7, verbose=False):
    # Calculate individual scores
    bigram_score = score(text, bigram_freq_dicts)
    word_score = word_based_score(text, word_freq_dict, total_words)
    
    # Normalize by text length
    text_length = max(len(text), 1)  # Avoid division by zero
    normalized_bigram = bigram_score / text_length
    normalized_word = word_score / text_length
    
    # Combine scores
    combined = alpha * normalized_bigram + (1 - alpha) * normalized_word
    
    if verbose and len(text.split()) > 5:
        known_words = sum(1 for w in text.split() if w in word_freq_dict)
        total_words_in_text = len(text.split())
        print(f"Bigram: {bigram_score:7.2f} | Word: {word_score:7.2f} | "
              f"Known: {known_words}/{total_words_in_text} | Combined: {combined:.4f}")
    
    return combined

In [ ]:
# Test
message_code = "WDYRDYLDQCSLR KTYDPYZ LXSKTYWDYQ U KTYZOITYGIDYCDYJSILPTYLDUDKOTYCIQYKQTP QPYODYPDRZTYCDQPDQCLDYODTYJL GIDRDQPTYSLH QKGIDTYCDTYMSKTDLKDTYCSIULKLYODTYADINYZSILYXKNDLYODYB ODKCSTJSZDYCDYOSMTJILKPDYCDYHSYPDLYHLYJDYIQDYOIDILYRSRDQP QYDYCDYJSQTJKDQJDYODYTSRRDKOYSIYDP KDQPYZOSQHYTYODTYRDIMODTYO YJF RMLDYODYPSIPYCSQPYWDYQDP KTYGIIQDYZDPKPDYZ LPKDYDPY YOKQTDQTKMKOKPDYCIGIDOYWDYLDPSILQ KTYUKPDYRIQKL"
word_freq_dict, total_words = build_word_frequency_dict(message_code)

def demonstrate_word_analysis():
    
    for description, test_text in test_cases:
        print(f"\n{description}:")
        print(f"Text: '{test_text}'")
        
        bigram_sc = score(test_text, bigram_stats)
        word_sc = word_based_score(test_text, word_freq_dict, total_words)
        combined_sc = combined_score(test_text, bigram_stats, word_freq_dict, total_words)
        
        words = test_text.split()
        known_count = sum(1 for w in words if w in word_freq_dict)
        
        print(f"  Bigram score: {bigram_sc:8.2f}")
        print(f"  Word score:   {word_sc:8.2f}")
        print(f"  Combined:     {combined_sc:8.4f}")
        print(f"  Known words:  {known_count}/{len(words)}")

## Extension Question #8: Frequency Analysis Adjustment

In this section we take a look at how far off the simple letter-frequency method actually is. 
The idea is to compare:

- the decode we get from the basic frequency mapping, and  
- the decode we get after running the Metropolis/bigram method.

By comparing the two, we can get a sense of how many cipher letters the simple method gets wrong, 
and how much adjustment the Metropolis method has to make.


In [ ]:
# Part 8 – checking how far off simple frequency analysis actually is
# The idea here is:
#   - get a decode from the basic letter-frequency method
#   - get a decode from the Metropolis/bigram method
#   - compare the two to see how many cipher letters change their mapping
#   - also measure how different the two decodes are position-by-position
#
# This gives us a rough idea of "how much" the simple approach needs to be fixed.

from collections import Counter

# quick helper: infer a mapping from (ciphertext, decoded_text)
def get_mapping_from_decoded(ciphertext, decoded_text):
    m = {}
    for c in character_list:
        targets = []
        for i, ch in enumerate(ciphertext):
            if ch == c and i < len(decoded_text):
                targets.append(decoded_text[i])
        if targets:
            # majority vote, nothing fancy
            m[c] = Counter(targets).most_common(1)[0][0]
        else:
            m[c] = c   # if not present in message, leave it mapped to itself
    return m

# compare how many letters map to the same plaintext
def compare_maps(m1, m2):
    same, diff = 0, 0
    for c in character_list:
        if m1[c] == m2[c]:
            same += 1
        else:
            diff += 1
    return same, diff

# check how many positions match in the two decoded strings
def positional_overlap(t1, t2):
    n = min(len(t1), len(t2))
    if n == 0:
        return 0.0
    matches = sum(1 for i in range(n) if t1[i] == t2[i])
    return matches / n


# ------- run the comparison on one ciphertext --------
# (can change this to any coded_message_* we want)
ciphertext = coded_message_02

# simple frequency-based decode
english_freqs = get_frequency(moby_new)
freq_map = get_letter_map_from_frequency(ciphertext, english_freqs, verbose=False)
decoded_freq = decode(ciphertext, freq_map)

# Metropolis/bigram decode
bigram_stats = all_bigram_freqs(moby_new, character_list)
decoded_metro = metropolis(ciphertext, character_list, bigram_stats,
                           N=20000, T=1.0, verbose=False)

# build inferred mappings for each decode
map_freq  = get_mapping_from_decoded(ciphertext, decoded_freq)
map_metro = get_mapping_from_decoded(ciphertext, decoded_metro)

same_letters, diff_letters = compare_maps(map_freq, map_metro)
pos_agree = positional_overlap(decoded_freq, decoded_metro)

print("Extension 8 – frequency vs Metropolis")
print("-------------------------------------")
print(f"Message length: {len(ciphertext)}")
print(f"Same letter mappings : {same_letters}")
print(f"Different mappings   : {diff_letters}")
print(f"Positional agreement : {pos_agree*100:.2f}%")

print("\n--- First 300 chars of frequency-based decode ---")
print(decoded_freq[:300])

print("\n--- First 300 chars of Metropolis decode ---")
print(decoded_metro[:300])
print("\n--------------------------------------------")

## Extension Question #9

Looking back at the Metropolis Algorithm, and analysing its progress over time, there are many different quantities that factor into its effectiveness. Here, we will look at extracting these quantities from the Metropolis function and analayse them in different ways.

We could do this using graphical representation. For example: a graph of the number of steps N against the values of the score function $S(m)$, but change we also the value of T to produce multiple different lines. We wuld plot these all on the same graph for an easy comparison.

After exploring several quantities and their behaviours, as well as how they affect other quantities, we will analyse them in further detail as a conclusion after experimenting with graphs.

**Here are the four quantities we are extracting and analysing from question 6:**
* The Score Function $S(m)$,
* The Temperature Parameter $T$,
* The Iteration Count,
* The Swapping Mechanism.

In [ ]:
#COLLECTING THE SCORE FUNCTION DATA NEEDED FOR THE GRAPHS:
y_values = score_history
x_values = list(range(len(y_values)))


text_length = len(message)

#Size of the figure
plt.figure(figsize=(10, 6))

#Plot the graph
plt.plot(x_values, y_values, label="Score Function $S(m)$")

#Label the axes
plt.xlabel("N")
plt.ylabel("$S(m)$")

#Show title and legend
plt.title("Analysis of the Score Function (with T = 1)")
plt.legend()
plt.show()

In [ ]:
#NOW WE HAVE 2 COMPARISON GRAPHS:

#TEMPERATURE:
#T = 10
final_text, T10_score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 20000, 10, verbose = False)
#T = 0.1
final_text, T01_score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 20000, 0.1, verbose = False)
#T = SMLLEST CASE
final_text, Tsmall_score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 20000, 0.00005, verbose = False)


#Size of the figure
plt.figure(figsize=(10, 6))

#Plot the graph
plt.plot(x_values, T10_score_history, label="Score Function $S(m)$ when T = 10")
plt.plot(x_values, T01_score_history, label="Score Function $S(m)$ when T = 0.1")
plt.plot(x_values, Tsmall_score_history, label="Score Function $S(m)$ when T = 0.00005")

#Label the axes
plt.xlabel("N")
plt.ylabel("$S(m)$")

#Show title and legend
plt.title("Analysis of the Score Function (with varying T values)")
plt.legend()
plt.show()

In [ ]:
#LENGTH (Setting the Temperature back to T = 1):
#LONG TEXT
final_text, long_score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 20000, 1, verbose = False)
#SHORT
#IMPORTING OUR SEVENTH CODED MESSAGE
notebook_path = 'CodedMessages.ipynb'
cell_index = 6

try:
    with open(notebook_path, 'r', encoding = 'utf-8') as f:
        nb = nbformat.read(f, as_version = 4)

    cell = nb.cells[cell_index]

    short_message = cell.source

except FileNotFoundError:
    print('Could not find file.')
except IndexError:
    print('Indexing Error.')

final_text, short_score_history = metropolis(short_message, character_list, all_bigram_freqs_dicts, 20000, 1, verbose = False)


#Size of the figure
plt.figure(figsize=(10, 6))

#Plot the graph
plt.plot(x_values, long_score_history, label="Score Function $S(m)$ with Longer Text")
plt.plot(x_values, short_score_history, label="Score Function $S(m)$ with Shorter Text")

#Label the axes
plt.xlabel("N")
plt.ylabel("$S(m)$")

#Show title and legend
plt.title("Analysis of the Score Function (with varying text lengths)")
plt.legend()
plt.show()

In [ ]:
#Since the Score Function for the shorter text plots a graph with a greater score value compared to the long text, we need to adjust the scaling for these values:
plt.figure(figsize=(10, 6))

#LENGTH (Setting the Temperature back to T = 1):
#LONG TEXT
final_text, long_score_history = metropolis(message, character_list, all_bigram_freqs_dicts, 20000, 1, verbose = False)

normalized_long = [score / len(message) for score in long_score_history]
plt.plot(normalized_long, label="Score Function $S(m)$ with Longer Text")

#SHORT
final_text, short_score_history = metropolis(short_message, character_list, all_bigram_freqs_dicts, 20000, 1, verbose = False)

normalized_short = [score / len(short_message) for score in short_score_history]
plt.plot(normalized_short, label="Score Function $S(m)$ with Shorter Text")


#Label the axes
plt.xlabel("N")
plt.ylabel("$S(m)$")

#Show title and legend
plt.title("Analysis of the Scaled Score Function (with varying text lengths)")
plt.legend()
plt.show()

* Imma yap here, i'm just too tired rn ngl

## <u>Results

* Final decoded messages with correct punctutation (and comparison to original message)
* Which method(s) did you use (from questions 2-6)? And why?
* How were these method(s) improved?

## <u>Conclusion

* General conclusion about the whole project, nothing too long